In [ ]:
from deep_translator import GoogleTranslator

# Read the original file
with open("../output/test-pipeline/test_de.txt", "r", encoding="utf-8") as file:
    original_text = file.read()

# Translate
translated_text = GoogleTranslator(source='auto', target='en').translate(original_text)

# Save the result
with open("../output/test-pipeline/translated.txt", "w", encoding="utf-8") as file:
    file.write(translated_text)


ModuleNotFoundError: No module named 'deep_translator'

In [1]:
from transformers import MarianMTModel, MarianTokenizer

model_name = 'Helsinki-NLP/opus-mt-de-en'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

with open("../output/test-pipeline/test_de.txt", "r", encoding="utf-8") as file:
    original_text = file.read()

inputs = tokenizer([original_text], return_tensors="pt", padding=True)
translated = model.generate(**inputs)
translated_text = tokenizer.decode(translated[0], skip_special_tokens=True)

print(translated_text)

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

After the end of the last Ice Age, the Roman Empire began to be inhabited by hunters and collectors who lived in caves and simple dwellings. Archeological finds prove human presence already about 10,000 years ago. Particularly important is the Venus of Willendorf, a statue dating back to around 29,500 years, which is considered one of the most famous artifacts of the Old Stone Age. With the Jungenstein Age began the sesshaft valorization of people, which was accompanied by the agriculture and the fieculture. First village structures arose and early forms of trade and crafts developed. In the Bronze Age, about 2,200 to 800 BC, more complex societies were established with the first power centers, which expressed themselves in fortified settlements. This early phase of Austria's history laid the foundation for later cultural developments and civilizations, which settled in this area. Keltenzeit From about 800 BC the territory of today's Austria was settled by the Kältische Stämen, which p

In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from nltk.tokenize import sent_tokenize
import torch, nltk

nltk.download("punkt")

tokenizer = T5Tokenizer.from_pretrained("t5-base")
model = T5ForConditionalGeneration.from_pretrained("t5-base").to("cuda" if torch.cuda.is_available() else "cpu")

with open("test_de.txt", "r", encoding="utf-8") as file:
    german_text = file.read().replace("\n", " ")

sentences = sent_tokenize(german_text, language="german")
translations = []

for sentence in sentences:
    prompt = f"translate German to English: {sentence}"
    input_ids = tokenizer.encode(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    output = model.generate(input_ids, max_new_tokens=100)
    translated = tokenizer.decode(output[0], skip_special_tokens=True)
    translations.append(translated)

final_translation = " ".join(translations)
print("\n--- Translated Text ---\n")
print(final_translation)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: 'test_de.txt'

In [3]:
from transformers import AutoTokenizer, Gemma3ForCausalLM, pipeline, Gemma3ForConditionalGeneration
from huggingface_hub import notebook_login
import torch
notebook_login()

# Load the instruction-tuned Gemma 3 model and tokenizer
model_id = "google/gemma-3-4b-it"

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto"
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = Gemma3ForCausalLM.from_pretrained(model_name, device="cuda", torch_dtype=torch.bfloat16)

# Create a text generation pipeline
translator = pipeline("text-generation", model_name, device="cuda", torch_dtype=torch.bfloat16)

with open("../output/test-pipeline/test_de.txt", "r", encoding="utf-8") as file:
    original_text = file.read()

# Construct the prompt for translation
prompt = f"Translate the following German text to English:\n\n{original_text}"

# Generate the translation
output = translator(prompt, max_new_tokens=100, do_sample=False)

# Extract and print the translated text
translated_text = output[0]['generated_text'].split('\n')[-1]
print("Translated text:", translated_text)


TypeError: argument of type 'NoneType' is not iterable

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from huggingface_hub import notebook_login
notebook_login()


#ngl alles unter 8b hat einfach ultra schlecht uebersetzt
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
with open("../output/test-pipeline/test_de.txt", "r", encoding="utf-8") as file:
    original_text = file.read()
    
messages = [
    {"role": "system", "content": "You are a translation chatbot that translates text from german to english!"},
    {"role": "user", "content": original_text},
]

print("mes")
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)
print("ids")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]
print("ter")
outputs = model.generate(
    input_ids,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
print("out")
response = outputs[0][input_ids.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


mes
ids
ter


KeyboardInterrupt: 